#### **Introduction to Principal Component Analysis** <br> Joshua Pajak, Ph.D. joshua.pajak@umassmed.edu

This notebook is intended to introduce the reader to the core components of principcal component analysis (PCA). While the context of the original writing is for students to understand how PCA applies to MD simulations, this notebook is constructed such that any student can hopefully glean insights into PCA.

For this notebook, we will use the immensely popular "iris" dataset which is standard fare for learning statsitics in the 21st century. 

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_iris


# Load the dataset
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = iris.target_names[iris.target]

# Display the first few rows
print("The Raw Data (First 5 Frames):")
display(df.head())

In [ ]:
# --- Plot the pairwise data correlations ---
sns.pairplot(df, hue='species', palette='viridis')
plt.suptitle("Raw Feature Correlations", y=1.02)
plt.show()

Ok, so we have our data. What do you notice? What do you think about the correlations between the data?

Let's introduce the concept of the **covariance matrix.** The covariance matrix is a matrix whose diagonal entries are the variance of our data $\sigma_{xx}^2$, while the off-diagonal entries are the covariance of our data $\sigma_{xy}^2$. 

To calculate the covariance matrix, we need to contextualize our data with respect to how far away it is from the mean. $$\sigma_{xy}^2=\frac{\sum{(x_{i}-\mu_{x})(y_{i}-\mu_y)}}{n-1}$$

An easy way to do this is to first center our data by subtracting the mean from each variable, as we will do in the cell below. 

In [ ]:
# --- Separate the numerical data from the labels ---
features = iris.feature_names
x_values = df[features]
labels = df['species']

# --- Perform the centering (subtracting the mean) ---
# This ensures that the 'center of mass' of our data is at zero
x_centered = x_values - x_values.mean()

# --- Combine them back into a new DataFrame for visualization ---
df_centered = pd.concat([x_centered, labels], axis=1)

In [ ]:
# --- Plot the centered data ---
sns.pairplot(df_centered, hue='species', palette='viridis')
plt.suptitle("Feature correlations deviation from mean", y=1.02)
plt.show()

Great! Now we need to construct our covariance matrix. I will show you how one would do it three ways:
1. Built-in Pandas functionality
2. Doing the summation brute-force
3. Doing the matrix multiplication

In [ ]:
# --- Calculate the covariance matrix using Pandas ---
# it's literally just one function call
cov_matrix = x_centered.cov()

print("The Covariance Matrix:")
display(cov_matrix)

# --- visualize a heatmap ---
plt.figure(figsize=(6, 6))
sns.heatmap(cov_matrix, annot=True, cmap='viridis', fmt=".2f")
plt.title("Covariance Heatmap")
plt.show()

### **Homework for the reader**
1. Do the diagonal entries and the off-diagonal entries tell us the "same thing"? Why aren't all diagonal entries high values? Shouldn't every variable be perfectly correlated itself?

In [ ]:
# --- Calculating covariance as brute force summation --- 
col1 = x_centered['sepal length (cm)']
col2 = x_centered['petal length (cm)']

# --- Multiply element-wise, sum, and divide by (n-1) ---
# this is the most inefficient, and yet intuitive way 
n = len(col1)
manual_cov = np.sum(col1 * col2) / (n - 1)

# --- Compare with the built-in pandas result --- 
pandas_cov = x_centered.cov().loc['sepal length (cm)', 'petal length (cm)']

print(f"Manual Covariance: {manual_cov:.4f}")
print(f"Pandas Covariance: {pandas_cov:.4f}")

In [ ]:
# --- The Matrix Math way: (X_transpose multiplied by X) / (n - 1)
# NumPy is THE way to do linear algebra in Python
# so we will convert from a PD DataFrame to a NumPy array
x_mat = x_centered.values

# NumPy arrays have a built-in method for transpose .T
# @ is the linear operator for matrix multiplication
manual_matrix = (x_mat.T @ x_mat) / (n - 1)

# --- Convert back to DataFrame for a pretty display ---
manual_matrix_df = pd.DataFrame(manual_matrix, index=features, columns=features)

print("Manual Covariance Matrix (calculated via multiplying NumPy arrays):")
display(manual_matrix_df)

print("\nDoes it match pandas .cov() exactly?")
print(np.allclose(manual_matrix, x_centered.cov()))

Great! So we have our covariance matrix. What properties does the covariance matrix have? How does it transform vectors?

In [ ]:
# --- Pick two features for 2D visualization ---
# We'll use Sepal Length and Sepal Width (indices 0 and 1)
feat_idx = [1, 3]
C_2d = cov_matrix.iloc[feat_idx, feat_idx].values
X_2d = x_centered.iloc[:, feat_idx].values

# --- Create a set of unit vectors (arrows) ---
# We'll create arrows pointing in different directions around a circle
angles = np.linspace(0, 2*np.pi, 8, endpoint=False)
unit_vectors = np.array([[np.cos(a), np.sin(a)] for a in angles])

# --- Transform these vectors using the Covariance Matrix (C * v) ---
# because our unit_vectors are stored as a matrix of row vectors
# we will use the mathematically equivalent operation of 
# v * C.T to make sure our dimensions work out ok.
transformed_vectors = np.dot(unit_vectors, C_2d.T) 

# --- Plotting the Transformation ---
fig, ax = plt.subplots(1, 2, figsize=(10, 6))

#--- Plot 1: The Original Unit Vectors ---
ax[0].scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.6, c='gray', label='Centered Data')
for i, v in enumerate(unit_vectors):
    ax[0].quiver(0, 0, v[0], v[1], color=plt.cm.tab10(i), scale=5, label=f'v{i}' if i<3 else "")
ax[0].set_title("Original Unit Vectors\n(Arbitrary Directions)")
ax[0].set_xlabel(iris.feature_names[0])
ax[0].set_ylabel(iris.feature_names[1])
ax[0].set_aspect('equal')

#--- Plot 2: The Transformed Vectors --- 
ax[1].scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.6, c='gray')
for i, v in enumerate(transformed_vectors):
    ax[1].quiver(0, 0, v[0], v[1], color=plt.cm.tab10(i), scale=5)
ax[1].set_title("Vectors after $C \\cdot v$\n(Stretched toward data variance)")
ax[1].set_xlabel(iris.feature_names[0])
ax[1].set_ylabel(iris.feature_names[1])
ax[1].set_aspect('equal')

plt.tight_layout()
plt.show()

In general, our covariance matrix will **stretch** and **rotate** any vector to a new vector which better matches the spread of our data. But some very special vectors will only be stretched. We call this vectors **eigenvectors $v$** and the amount that they are scaled by as **eigenvalues $\lambda$.** 

That is, $v$ is an eigenvector of matrix $A$ if $$A \cdot v = \lambda v$$ where $\lambda$ is a scalar.

Because eigenvectors are only stretched by the covariance matrix (and not rotated), these vectors **already point in a direction which describes the variance of our data.**

We call the eigenvectors of the covariance matrix the **principal components** of our data.

In [ ]:
# --- calculate eigenvectors and eigenvalues for 2D case ---
eigvals_2d, eigvecs_2d = np.linalg.eig(C_2d)

# Note: eigvecs_2d columns are the eigenvectors. 
# Let's extract them as row vectors for easier plotting
ev1 = eigvecs_2d[:, 0]
ev2 = eigvecs_2d[:, 1]

# --- Use the covariance matrix as a linear operator on our eigenvectors ---
ev1_transformed = C_2d @ ev1
ev2_transformed = C_2d @ ev2
# note that @ and np.dot are equivalent operations but np.dot is faster

# --- Plot the result ---
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# Define colors for the two eigenvectors
colors = ['red', 'blue']

for i in range(2):
    # Plot the centered data as background
    ax[i].scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.2, c='gray')

# Plot 1: Before Transformation
ax[0].set_title("Original eigenvectors\n(The 'Principal' Directions)")
ax[0].quiver(0, 0, ev1[0], ev1[1], color=colors[0], scale=5, alpha=0.5, linestyle='--')
ax[0].quiver(0, 0, ev2[0], ev2[1], color=colors[1], scale=5, alpha=0.5, linestyle='--')

# Plot 2: After Transformation
ax[1].quiver(0, 0, ev1_transformed[0], ev1_transformed[1], color=colors[0], scale=5, alpha=0.5, linestyle='--')
ax[1].quiver(0, 0, ev2_transformed[0], ev2_transformed[1], color=colors[1], scale=5, alpha=0.5, linestyle='--')
ax[1].set_title("Transformed eigenvectors\n(Scaled by eigenvalues, but NOT Rotated)")


plt.tight_layout()
plt.show()

# ---  Print the Verification ---
print(f"Eigenvalue 1: {eigvals_2d[0]:.4f}")
print(f"Stretch factor of EV1: {np.linalg.norm(ev1_transformed):.4f}")
print(f"\nEigenvalue 2: {eigvals_2d[1]:.4f}")
print(f"Stretch factor of EV2: {np.linalg.norm(ev2_transformed):.4f}")

In general, we will have as many Principal Components as we have dimensions in our data. But I thought we said that PCA was a "dimensionality reduction" technique? It is! Because generally only the first few PCs actually matter. We can plot a "scree" plot to visualize this. 

In [ ]:
# --- Find eigenvalues and eigenvectors of the entire dataset ---
# again, NumPy is THE way to do this in Python
eigenvalues, eigenvectors = np.linalg.eig(manual_matrix)

# Let's put them in a DataFrame so we can see which original features contribute to our PCs
eig_df = pd.DataFrame(eigenvectors, index=features, columns=['PC1', 'PC2', 'PC3', 'PC4'])

print("The Eigenvectors (The recipes for each Principal Component):")
display(eig_df)

In [ ]:
# --- Sort the eigenvalues and eigenvectors ---
# np.linalg.eig doesn't always return them in order and we want the biggest first
idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# --- Calculate the Percentage of Variance Explained ---
total_variance = np.sum(eigenvalues)
var_exp = (eigenvalues / total_variance) * 100
cum_var_exp = np.cumsum(var_exp)

# --- Plot the Scree Plot ---
plt.figure(figsize=(9, 5))

# --- Plot the individual variance (Bar chart) ---
plt.bar(range(1, 5), var_exp, align='center', 
        label='Individual Explained Variance', color='gray')

# --- Plot the cumulative variance (Step plot) ---
plt.step(range(1, 5), cum_var_exp, where='mid', 
         label='Cumulative Explained Variance', color='red', marker='o')

# --- Formatting the plot ---
plt.ylabel('Percentage of Variance Explained')
plt.xlabel('Principal Component Index')
plt.title('Scree Plot: How much "Signal" is in each PC?')
plt.xticks(range(1, 5), ['PC1', 'PC2', 'PC3', 'PC4'])
plt.ylim(0, 110)
plt.legend(loc='best')
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# --- Print these results ---
print("Variance Explained per Component:")
for i, (v, c) in enumerate(zip(var_exp, cum_var_exp)):
    print(f"PC{i+1}: {v:6.2f}%  (Cumulative: {c:6.2f}%)")

So now we see that although we have four-dimensional data, we only need **one** PC to capture 92% of our variation! And if we halve the dimensionality of our data from four-dimensions to two-PCs, we cpature a whopping **97.8%** of the variation in our data!

This is the power of PCA. 

### **Homework for the reader**
2. What are some limitations of PCA? 
3. Code up a way to project our centered data onto PC-space
4. Code up a way to calculate PCA using the sklearn library directly from our Pandas DataFrame